In [2]:
import matplotlib.pyplot as plt #viz #GUI manager
import seaborn as sns #viz #plotly is another package
import datetime
import pandas as pd
import numpy as np
from pandas import Grouper #groupby
#statistical data exploration, conducting statistical tests, and estimation of different statistical models
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf #autocorrelation plot
from statsmodels.tsa.api import SimpleExpSmoothing
from statsmodels.tsa.holtwinters import ExponentialSmoothing # double and triple exponential smoothing
from pandas.plotting import autocorrelation_plot #autocorrelation plot
from statsmodels.graphics.gofplots import qqplot #residual diagnostics
from sklearn.metrics import mean_squared_error #accuracy metrics
from math import sqrt
from sklearn.metrics import mean_absolute_error #accuracy metrics

from random import gauss #create gaussian white noise
from random import seed
from pandas import Series

from statsmodels.tsa.stattools import adfuller # Augmented Dickey Fuller test for testing stationarity

from statsmodels.tsa.arima_model import ARIMA #for manual ARIMA

import pmdarima as pm #auto arima
import scipy.io as sio
import os
from pathlib import Path
import warnings


## Data Loading

In [4]:
base_path = (
    "/Users/jiangruitong/Library/CloudStorage/"
    "GoogleDrive-ruitongj@andrew.cmu.edu/Shared drives/"
    "NML_NHP/Cynos PVT Data/Session Info/{subject}/Additional Performance Data"
)
# Helpers function
def _parse_filename(fname:str)-> dict:
    """
    Parse the filename to extract file_month,file_day, session, subject_initial
    Return None on parse failure
    """
    stem = Path(fname).stem
    parts = stem.split("_")
    if len(parts) !=4:
        warnings.warn(f"Unexpected filename format(expected 4 parts): {fname}")
        return None
    return {
        'file_month': parts[0],
        'file_day': int(parts[1]),
        'session': parts[2],
        'subject_initial': parts[3]
    }
def _read_session_csv(filepath:str) -> tuple[pd.DataFrame,dict]:
    """
    Read a single PVT session CSV file and return a trial-level DataFrame
    Row 0 contains both the session summary fields and the first trial RT
    Rows 1 to N contain the remaining trial RTs (summary fields are empty)
    The key RT column is 'Correct Response Latency_Duration'
    """
    df = pd.read_csv(filepath)
    # Extract session-level metadata from row 0
    session_meta = {
        'datetime_str': df['Date/Time'].iloc[0],
        'session_time_s': df['Summary - Session Time'].iloc[0],
        'n_correct':df['Summary - Correct Trials'].iloc[0],
        'n_omitted':df['Summary - Omitted Trials'].iloc[0],
        'n_missed':df['Summary - Missed Trials'].iloc[0],
        'n_iti_touches':df['Summary - ITI Touches'].iloc[0]
    }
    rt = df['Correct Response Latency_Duration'].values
    trials = pd.DataFrame({
        'trial_num':np.arange(1, len(rt)+1),
        'rt': rt
    })
    return trials, session_meta

# Main compilation function
def compile_pvt_data(subjects:list[str]) -> tuple[dict[str,pd.DataFrame],dict[str,pd.DataFrame]]:
    """
    Compile PVT data for the given list of subjects.
    Input: subjects - list of subject names

    Returns:
    trials_dict: dict[str,pd.DataFrame]
        One row per trial. Columns: subject, date, session, trial_num, rt
    
        session_dict: dict[str,pd.DataFrame]
        One row per session. Columns: subject, date, session, n_correct, n_omitted, n_missed,n_iti_touches
    """
    trials_dict = {}
    sessions_dict={}
    for subj in subjects:
        folder = base_path.format(subject=subj)
        if not os.path.isdir(folder):
            warnings.warn(f"Folder not found, skipping:{folder}")
            continue
        csv_files = sorted(f for f in os.listdir(folder) if f.endswith('.csv'))
        if not csv_files:
            warnings.warn(f"No CSV files found in {folder}")
            continue
        print(f"[{subj}] Found {len(csv_files)} session files.")
        subj_trials = []
        subj_sessions = []
    
        for fname in csv_files:
            fpath = os.path.join(folder, fname)
            finfo = _parse_filename(fname)
            if finfo is None:
                continue
            try:
                trials,meta = _read_session_csv(fpath)
            except Exception as e:
                warnings.warn(f"Failed to read {fpath}: {e}")
                continue
        # Derive date fields from CSV's Date/Time column
            dt = pd.to_datetime(
                meta['datetime_str'], format='%d/%m/%Y %H:%M:%S',errors='coerce'
            )
            date_fields = {
                'date':dt.date() if pd.notna(dt) else None,
                'month': dt.month if pd.notna(dt) else None
            }
        # Trial rows
            trials['subject'] = subj
            trials['date']=date_fields['date']
            trials['session']=finfo['session']
            subj_trials.append(trials)
        # Session row
            subj_sessions.append({
                'subject':subj,
                'date':date_fields['date'],
                'session':finfo['session'],
                'n_correct':meta['n_correct'],
                'n_omitted':meta['n_omitted'],
                'n_missed':meta['n_missed'],
                'n_iti_touches':meta['n_iti_touches']
            })
        if not subj_trials:
            warnings.warn(f"No valid data for {subj}")
            continue

        # Assemble and order columns
        df_trials = pd.concat(subj_trials, ignore_index=True)
        df_trials = df_trials[['subject','date','session','rt']]
        df_sessions = pd.DataFrame(subj_sessions)
        df_sessions = df_sessions[['subject','date','session',
                                   'n_correct','n_omitted','n_missed','n_iti_touches']]
        trials_dict[subj]=df_trials
        sessions_dict[subj]=df_sessions
        print(f"-> {len(df_trials)} trials, {len(df_sessions)} sessions")
    return trials_dict, sessions_dict

trials,sessions = compile_pvt_data(['Virgil','Baldface','Goku','Beans'])
df_v_rt = trials['Virgil']
df_g_rt = trials['Goku']
df_bf_rt = trials['Baldface']
df_b_rt = trials['Beans']
    

[Virgil] Found 180 session files.
-> 33161 trials, 180 sessions
[Baldface] Found 201 session files.
-> 29339 trials, 201 sessions
[Goku] Found 198 session files.
-> 25568 trials, 198 sessions
[Beans] Found 189 session files.
-> 25117 trials, 189 sessions


## Generate a summary of the reaction time data